# Catalyst Center - Switches and Hubs Viewer Tutorial

This notebook demonstrates how to interact with Cisco Catalyst Center's REST API to retrieve and display network devices. We'll walk through each step, explaining the API calls and authentication process.

## Overview
We'll accomplish the following:
1. **Authentication**: Get an API token using HTTP Basic Auth
2. **Query Devices**: Retrieve all switches and hubs using the token
3. **Display Results**: Format and show device information

---

## Step 1: Import Dependencies and Suppress Warnings

First, let's import the necessary libraries and suppress SSL warnings (since we're using `verify=False` for demo purposes).

In [1]:
import os
import warnings

# Suppress urllib3 SSL warnings when using verify=False
warnings.filterwarnings("ignore", message=".*urllib3.*", category=Warning)

import requests
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv

### Libraries Explained:
- **`requests`**: HTTP library for making REST API calls
- **`HTTPBasicAuth`**: Helper for HTTP Basic Authentication
- **`dotenv`**: Loads environment variables from `.env` file (keeps credentials secure)
- **`warnings`**: Allows us to suppress SSL certificate warnings

## Step 2: Load Configuration from Environment

Load credentials and connection details from the `.env` file. This keeps sensitive information out of your code.

In [2]:
# Load environment variables from .env file
load_dotenv()

# Extract configuration
BASE_URL = os.getenv("CATALYST_BASE_URL")  # e.g., https://catalyst-center.example.com
USERNAME = os.getenv("CATALYST_USERNAME")  # API username
PASSWORD = os.getenv("CATALYST_PASSWORD")  # API password
VERSION  = os.getenv("CATALYST_VERSION")   # API version (optional)

print(f"✓ Configuration loaded")
print(f"  Base URL: {BASE_URL}")
print(f"  Username: {USERNAME}")

✓ Configuration loaded
  Base URL: https://10.48.90.165
  Username: admin


### Configuration Variables:
- **`BASE_URL`**: The root URL of your Catalyst Center instance
- **`USERNAME`** / **`PASSWORD`**: Credentials for API access
- **`VERSION`**: API version (not used in this example, but available for version-specific endpoints)

**Security Note**: Never hardcode credentials! Always use environment variables or secure vaults.

## Step 3: Authentication - Get API Token

Catalyst Center uses token-based authentication. We first authenticate with username/password to receive a token, then use that token for subsequent API calls.

### API Endpoint: `/dna/system/api/v1/auth/token`
- **Method**: POST
- **Authentication**: HTTP Basic Auth (username + password)
- **Returns**: JSON object with `Token` field

In [3]:
def get_auth_token():
    """
    Authenticate with Catalyst Center and return an X-Auth-Token.
    
    This function:
    1. Sends POST request to /auth/token endpoint
    2. Includes username/password via HTTP Basic Auth
    3. Disables SSL verification (verify=False)
    4. Extracts and returns the token from JSON response
    """
    url = f"{BASE_URL}/dna/system/api/v1/auth/token"
    
    print(f"📡 Requesting authentication token...")
    print(f"   URL: {url}")
    
    response = requests.post(
        url, 
        auth=HTTPBasicAuth(USERNAME, PASSWORD), 
        verify=False  # Skip SSL cert verification (use cautiously!)
    )
    
    # Raise exception if request failed (4xx or 5xx status codes)
    response.raise_for_status()
    
    token = response.json()["Token"]
    print(f"✓ Token received: {token[:20]}...")
    
    return token

# Get the authentication token
token = get_auth_token()

📡 Requesting authentication token...
   URL: https://10.48.90.165/dna/system/api/v1/auth/token
✓ Token received: eyJ0eXAiOiJKV1QiLCJh...


### Authentication Flow Explained:

1. **POST Request**: We send credentials to the authentication endpoint
2. **Basic Auth**: Username/password are Base64-encoded in the `Authorization` header
3. **Token Response**: Server validates credentials and returns a JWT-style token
4. **Token Usage**: This token must be included in all subsequent API requests via `X-Auth-Token` header

**Token Lifecycle**:
- Tokens typically expire after 1 hour
- Production apps should implement token refresh logic
- Store tokens securely; don't log or expose them

## Step 4: Query Network Devices

Now that we have a valid token, let's retrieve all devices in the "Switches and Hubs" family.

### API Endpoint: `/dna/intent/api/v1/network-device`
- **Method**: GET
- **Authentication**: X-Auth-Token header
- **Query Parameters**: 
  - `family`: Filter by device family (e.g., "Switches and Hubs", "Routers", "Wireless Controller")
- **Returns**: JSON with `response` array containing device objects

In [4]:
def get_switches(token):
    """
    Retrieve all devices in the 'Switches and Hubs' family.
    
    This function:
    1. Constructs the network-device API endpoint URL
    2. Includes the auth token in request headers
    3. Filters results using 'family' query parameter
    4. Returns the list of device objects
    """
    url = f"{BASE_URL}/dna/intent/api/v1/network-device"
    
    # Token must be passed in X-Auth-Token header
    headers = {"X-Auth-Token": token}
    
    # Filter by device family
    params = {"family": "Switches and Hubs"}
    
    print(f"\n📡 Querying network devices...")
    print(f"   URL: {url}")
    print(f"   Filter: family='Switches and Hubs'")
    
    response = requests.get(
        url, 
        headers=headers, 
        params=params, 
        verify=False
    )
    
    response.raise_for_status()
    
    devices = response.json().get("response", [])
    print(f"✓ Retrieved {len(devices)} device(s)")
    
    return devices

# Query the devices
devices = get_switches(token)


📡 Querying network devices...
   URL: https://10.48.90.165/dna/intent/api/v1/network-device
   Filter: family='Switches and Hubs'
✓ Retrieved 2 device(s)


### Device Query Explained:

**Headers**:
- `X-Auth-Token`: Required for authentication (the token we obtained earlier)

**Query Parameters**:
- `family`: Filters devices by type
  - Other options: "Routers", "Wireless Controller", "Unified AP"
  - You can also filter by: `hostname`, `managementIpAddress`, `macAddress`, `serialNumber`, etc.

**Response Structure**:
```json
{
  "response": [
    {
      "hostname": "switch-01",
      "upTime": "123 days, 4:56:12.34",
      "managementIpAddress": "10.0.0.1",
      "serialNumber": "FCW1234ABCD",
      "softwareVersion": "17.3.1",
      ... (many more fields)
    }
  ]
}
```

## Step 5: Inspect a Sample Device Object

Let's examine the structure of a device object to see what data is available.

In [5]:
# Show the first device (if any exist)
if devices:
    print("Sample device object (first 10 fields):\n")
    sample_device = devices[0]
    
    # Display key fields
    for i, (key, value) in enumerate(sample_device.items()):
        if i < 10:  # Show first 10 fields
            print(f"  {key}: {value}")
        else:
            print(f"\n  ... and {len(sample_device) - 10} more fields")
            break
else:
    print("No devices found.")

Sample device object (first 10 fields):

  description: Cisco IOS Software [Dublin], Catalyst L3 Switch Software (CAT9K_IOSXE), Version 17.12.4, RELEASE SOFTWARE (fc3) Technical Support: http://www.cisco.com/techsupport Copyright (c) 1986-2024 by Cisco Systems, Inc. Compiled Tue 23-Jul-24 09:40 by mcpre netconf enabled
  type: Cisco Catalyst 9500 Switch
  lastUpdateTime: 1782848561647
  macAddress: 70:18:a7:61:b3:00
  deviceSupportLevel: Supported
  softwareType: IOS-XE
  softwareVersion: 17.12.4
  serialNumber: FCW2245F1LQ
  collectionInterval: Global Default
  dnsResolvedManagementAddress: 10.48.90.171

  ... and 43 more fields


### Common Device Fields:

- **`hostname`**: Device hostname/name
- **`upTime`**: How long the device has been running
- **`managementIpAddress`**: IP address for management
- **`serialNumber`**: Hardware serial number
- **`platformId`**: Device model (e.g., "C9300-24UX")
- **`softwareVersion`**: IOS/IOS-XE version
- **`role`**: Device role (ACCESS, DISTRIBUTION, CORE, etc.)
- **`reachabilityStatus`**: Current reachability (Reachable, Unreachable)
- **`family`**: Device family/type
- **`type`**: Detailed device type

## Step 6: Display Results in a Formatted Table

Finally, let's format and display the device information in a clean, readable table.

In [6]:
def print_devices(devices):
    """
    Print devices in a formatted table.
    
    Displays:
    - Hostname (left-aligned, 50 characters wide)
    - Uptime (right-aligned)
    """
    print(f"\n{'HOSTNAME':<50} {'UPTIME'}")
    print("-" * 75)
    
    for device in devices:
        hostname = device.get("hostname", "N/A")
        uptime   = device.get("upTime", "N/A")
        print(f"{hostname:<50} {uptime}")
    
    print("-" * 75)
    print(f"Total devices: {len(devices)}\n")

# Display the devices
print_devices(devices)


HOSTNAME                                           UPTIME
---------------------------------------------------------------------------
CSS-DNA-POD2-BORDER1.bru-css-dna-pod2.cisco.com    7 days, 0:17:47.48
CSS-DNA-POD2-EDGE2.bru-css-pod2.cisco.com          6 days, 20:01:55.31
---------------------------------------------------------------------------
Total devices: 2



### Display Formatting:

- **`{hostname:<50}`**: Left-align hostname in 50-character field
- **`device.get("field", "N/A")`**: Safely retrieve field with default value
- **`.get()` vs `[]`**: Using `.get()` prevents KeyError if field is missing

---

## Step 7: Pagination — Controlling How Many Devices You Retrieve

By default, the API may cap results at 500 devices. In large networks (hundreds or thousands of devices), you **must** use pagination to retrieve everything reliably.

### Why Pagination Matters

| Scenario | Without Pagination | With Pagination |
|----------|-------------------|-----------------|
| **10 devices** | ✅ Works fine | ✅ Works fine |
| **600 devices** | ❌ Silent truncation at 500 | ✅ All 600 retrieved |
| **5,000 devices** | ❌ Misses 4,500 devices | ✅ All retrieved in batches |

### Two Key Parameters

- **`offset`** — where to start (1-indexed). `offset=1` starts at the first device.
- **`limit`** — how many records to return per request. Max is `500`.

```
offset=1,  limit=10  →  devices 1–10   (page 1)
offset=11, limit=10  →  devices 11–20  (page 2)
offset=21, limit=10  →  devices 21–30  (page 3)
```

### Example 1: Retrieve the First 10 Devices

We pass `offset=1` and `limit=10` as query parameters.

In [ ]:
def get_devices_paginated(token, offset=1, limit=10):
    """
    Retrieve network devices with pagination control.

    Args:
        token  : X-Auth-Token from authentication step
        offset : First record to return (1-indexed, default 1)
        limit  : Number of records to return (default 10, max 500)
    Returns:
        list of device dicts
    """
    url     = f"{BASE_URL}/dna/intent/api/v1/network-device"
    headers = {"X-Auth-Token": token}
    params  = {"offset": offset, "limit": limit}

    response = requests.get(url, headers=headers, params=params, verify=False)
    response.raise_for_status()
    return response.json().get("response", [])


# --- Page 1: first 10 devices ---
page1 = get_devices_paginated(token, offset=1, limit=10)
print(f"Page 1 — devices 1-10 — retrieved: {len(page1)}")

print(f"\n{'HOSTNAME':<40} {'IP ADDRESS':<20} {'FAMILY'}")
print("-" * 80)
for d in page1:
    print(f"{d.get('hostname','N/A'):<40} {d.get('managementIpAddress','N/A'):<20} {d.get('family','N/A')}")
print("-" * 80)

### Example 2: Retrieve the Next 10 Devices (Page 2)

Increment `offset` by the page size to get the next batch.

> **Rule of thumb**: `offset` for page N = `(N-1) * limit + 1`
> - Page 1 → offset = 1
> - Page 2 → offset = 11
> - Page 3 → offset = 21

In [ ]:
# --- Page 2: next 10 devices ---
page2 = get_devices_paginated(token, offset=11, limit=10)
print(f"Page 2 — devices 11-20 — retrieved: {len(page2)}")

print(f"\n{'HOSTNAME':<40} {'IP ADDRESS':<20} {'FAMILY'}")
print("-" * 80)
for d in page2:
    print(f"{d.get('hostname','N/A'):<40} {d.get('managementIpAddress','N/A'):<20} {d.get('family','N/A')}")
print("-" * 80)

print(f"\n💡 Tip: if len(page2) < 10, you have reached the last page.")

### Example 3: Retrieve ALL Devices Automatically

For production use, loop through pages until the API returns fewer records than `page_size` — that signals the last page.

In [ ]:
def get_all_devices(token, page_size=50):
    """
    Retrieve ALL network devices by automatically iterating through pages.

    The loop stops when the API returns fewer records than page_size,
    which indicates there are no more pages.

    Args:
        token     : X-Auth-Token
        page_size : Records per request (default 50, max 500)
    Returns:
        list of all device dicts
    """
    all_devices = []
    offset = 1

    while True:
        print(f"  Fetching offset={offset}, limit={page_size}...")
        page = get_devices_paginated(token, offset=offset, limit=page_size)

        if not page:          # empty response → no more data
            break

        all_devices.extend(page)
        print(f"  → Got {len(page)} devices  (running total: {len(all_devices)})")

        if len(page) < page_size:   # partial page → last page reached
            break

        offset += page_size         # advance to next page

    return all_devices


print("🔄 Fetching ALL devices with automatic pagination (page_size=50)...\n")
all_devices = get_all_devices(token, page_size=50)
print(f"\n✅ Done — total devices retrieved: {len(all_devices)}")

---

## Step 8: Caching — Avoid Repeating Expensive API Calls

Calling `get_all_devices()` on every refresh hits the API many times. A simple **in-memory cache** stores the result and reuses it for 5 minutes:

```
First call  (t=0s)   → Cache MISS  → makes API calls → stores result
Second call (t=30s)  → Cache HIT   → returns stored result instantly ⚡
Third call  (t=290s) → Cache HIT   → still valid (< 300 s)
Fourth call (t=310s) → Cache EXPIRED → fetches fresh data again
```

### Real-world Impact

| Scenario | Without cache | With 5-min cache |
|----------|--------------|------------------|
| Dashboard refreshes every 30 s | **120 API calls / hour** | **12 API calls / hour** |
| Savings | — | **90 % reduction** |


In [ ]:
import time

# In-memory cache storage
_cache   = {}
CACHE_TTL = 300  # seconds (5 minutes)


def get_all_devices_cached(token, page_size=50, cache_ttl=CACHE_TTL):
    """
    Same as get_all_devices() but caches results for cache_ttl seconds.

    Returns a dict with:
        devices   - list of device dicts
        cached    - True if result came from cache (no API call made)
        cache_age - seconds since the cache was last populated
    """
    cache_key    = "all_devices"
    current_time = time.time()

    if cache_key in _cache:
        age = current_time - _cache[cache_key]["timestamp"]
        if age < cache_ttl:
            print(f"✅ Cache HIT  — age: {age:.1f}s  (TTL: {cache_ttl}s)  ⚡ No API calls!")
            return {"devices": _cache[cache_key]["data"], "cached": True, "cache_age": age}
        print(f"⏰ Cache EXPIRED — age: {age:.1f}s > TTL: {cache_ttl}s")
    else:
        print("❌ Cache MISS — fetching from API...")

    devices = get_all_devices(token, page_size)
    _cache[cache_key] = {"data": devices, "timestamp": time.time()}
    print(f"💾 Cached {len(devices)} devices for {cache_ttl}s")
    return {"devices": devices, "cached": False, "cache_age": 0}


# --- Demonstrate the cache ---
print("=" * 60)
print("CALL 1  (t=0s) — expect cache MISS")
print("=" * 60)
r1 = get_all_devices_cached(token)
print(f"Devices: {len(r1['devices'])}  |  From cache: {r1['cached']}\n")

time.sleep(2)

print("=" * 60)
print("CALL 2  (t≈2s) — expect cache HIT")
print("=" * 60)
r2 = get_all_devices_cached(token)
print(f"Devices: {len(r2['devices'])}  |  From cache: {r2['cached']}  |  Age: {r2['cache_age']:.1f}s")

---

## Complete Workflow Summary

Here's the entire process in one code cell for easy execution:

In [ ]:
# Complete workflow - All in one

# 1. Authentication
token = get_auth_token()

# 2. Query devices
devices = get_switches(token)

# 3. Display results
print_devices(devices)

---

## Exercise: Try These Variations

Now that you understand the API calls, try modifying the code:

### 1. Query Different Device Families

In [ ]:
# Try querying routers instead
def get_devices_by_family(token, family_name):
    url = f"{BASE_URL}/dna/intent/api/v1/network-device"
    headers = {"X-Auth-Token": token}
    params = {"family": family_name}
    
    response = requests.get(url, headers=headers, params=params, verify=False)
    response.raise_for_status()
    return response.json().get("response", [])

# Uncomment to try:
# routers = get_devices_by_family(token, "Routers")
# print_devices(routers)

### 2. Display Additional Fields

In [ ]:
# Show more device details
def print_detailed_devices(devices):
    for device in devices:
        print(f"\n{'='*60}")
        print(f"Hostname: {device.get('hostname', 'N/A')}")
        print(f"IP Address: {device.get('managementIpAddress', 'N/A')}")
        print(f"Serial Number: {device.get('serialNumber', 'N/A')}")
        print(f"Platform: {device.get('platformId', 'N/A')}")
        print(f"Software: {device.get('softwareVersion', 'N/A')}")
        print(f"Uptime: {device.get('upTime', 'N/A')}")
        print(f"Status: {device.get('reachabilityStatus', 'N/A')}")

# Uncomment to try:
# print_detailed_devices(devices)

### 3. Filter by Hostname

In [ ]:
# Query specific device by hostname
def get_device_by_hostname(token, hostname):
    url = f"{BASE_URL}/dna/intent/api/v1/network-device"
    headers = {"X-Auth-Token": token}
    params = {"hostname": hostname}
    
    response = requests.get(url, headers=headers, params=params, verify=False)
    response.raise_for_status()
    return response.json().get("response", [])

# Uncomment and replace with actual hostname:
# specific_device = get_device_by_hostname(token, "switch-01")
# print_devices(specific_device)

---

## Additional API Resources

### Other Useful Catalyst Center API Endpoints:

- **Get Device Details**: `GET /dna/intent/api/v1/network-device/{id}`
- **Get Device Config**: `GET /dna/intent/api/v1/network-device/{id}/config`
- **Get Interface Info**: `GET /dna/intent/api/v1/interface`
- **Get Site Hierarchy**: `GET /dna/intent/api/v1/site`
- **Get Client Health**: `GET /dna/intent/api/v1/client-health`

### API Documentation:
Visit your Catalyst Center instance at: `https://{your-catalyst-center}/dna/platform/app/consumer-portal/developer-toolkit/apis`

---

## Best Practices

1. **Error Handling**: Always use `try/except` blocks in production
2. **Token Management**: Implement token refresh before expiration
3. **Rate Limiting**: Respect API rate limits (check response headers)
4. **SSL Verification**: Use proper SSL certificates in production (`verify=True`)
5. **Logging**: Log API calls for debugging and audit purposes
6. **Pagination**: Handle large result sets with pagination parameters
7. **Async Operations**: Some API calls return task IDs - poll for completion